# 05_v2 Modeling Dataset Construction

Create final leakage-controlled modeling tables and feature set definitions only. No model training, SHAP, segmentation, or simulation is performed.


In [1]:
import csv
import json
from collections import Counter
from pathlib import Path

def find_project_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "_data" / "01_raw" / "Membership.csv").exists() and (candidate / "park.ingyeom" / "reports" / "data" / "02_v2_preprocessing_policy" / "membership_v2_preprocessed.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root from notebook execution directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
STAGE02 = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "02_v2_preprocessing_policy" / "membership_v2_preprocessed.csv"
USAGE_W13 = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "03_v2_usage_feature_engineering" / "usage_features_v2_w1_3.csv"
USAGE_W14 = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "03_v2_usage_feature_engineering" / "usage_features_v2_w1_4.csv"
CONTENT_W13 = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "04_v2_content_feature_engineering" / "content_features_v2_w1_3.csv"
CONTENT_W14 = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "04_v2_content_feature_engineering" / "content_features_v2_w1_4.csv"
DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "05_v2_modeling_dataset"
TABLE_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "tables" / "05_v2_modeling_dataset"
DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = [STAGE02, USAGE_W13, USAGE_W14, CONTENT_W13, CONTENT_W14]
RAW_FILES = [PROJECT_ROOT / "_data" / "01_raw" / name for name in ["Membership.csv", "User_Mapping.csv", "View_History.csv", "Movie_Master.csv"]]
FORBIDDEN_FEATURES = {"USER_KEY", "USER_NUM", "MOVIE_NUM", "movie_title", "reg_date", "end_date", "duration_days", "watch_date", "watch_day", "is_repurchase", "membership_row_id"}
FORBIDDEN_SUBSTRINGS = ["raw_calendar", "days_to_end", "days_since_last_watch_to_end"]
METADATA_COLUMNS = ["membership_row_id", "USER_KEY"]
TARGET_COLUMN = "is_repurchase"
MEMBERSHIP_FEATURES_WITH_CHURN = ["price", "product_code", "max_screen", "is_promotion", "is_user_verified", "gender", "age", "payment_device", "billing_method", "is_churn_prevented"]
MEMBERSHIP_FEATURES_WITHOUT_CHURN = [c for c in MEMBERSHIP_FEATURES_WITH_CHURN if c != "is_churn_prevented"]

def snapshot(paths):
    return {str(p): {"size": p.stat().st_size, "mtime_ns": p.stat().st_mtime_ns} for p in paths if p.exists()}

def rel(path):
    return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")

def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        return reader.fieldnames or [], list(reader)

def write_csv(path, rows, fields):
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, "") for field in fields})

def write_json(path, payload):
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

raw_before = snapshot(RAW_FILES)
inputs_before = snapshot(INPUTS)

m_cols, membership = read_csv(STAGE02)
u13_cols, usage13 = read_csv(USAGE_W13)
u14_cols, usage14 = read_csv(USAGE_W14)
c13_cols, content13 = read_csv(CONTENT_W13)
c14_cols, content14 = read_csv(CONTENT_W14)

def index_by_mid(rows):
    return {row["membership_row_id"]: row for row in rows}

membership_by_mid = index_by_mid(membership)
usage13_by_mid = index_by_mid(usage13)
usage14_by_mid = index_by_mid(usage14)
content13_by_mid = index_by_mid(content13)
content14_by_mid = index_by_mid(content14)

def infer_role(col, feature_sets):
    if col == TARGET_COLUMN:
        return "target"
    if col in METADATA_COLUMNS:
        return "metadata"
    if col in feature_sets:
        return "candidate_feature"
    return "excluded_audit_or_source_column"

def has_forbidden_feature(col):
    if col in FORBIDDEN_FEATURES:
        return True
    return any(token in col for token in FORBIDDEN_SUBSTRINGS)

def clean_feature_list(cols):
    return [c for c in cols if c not in {"membership_row_id"} and not has_forbidden_feature(c)]

usage_w1_3_features = clean_feature_list([c for c in u13_cols if c.startswith("w1_3_")])
usage_w1_4_features = clean_feature_list([c for c in u14_cols if c.startswith("w1_4_")])
content_w1_3_features = clean_feature_list([c for c in c13_cols if c.startswith("w1_3_")])
content_w1_4_features = clean_feature_list([c for c in c14_cols if c.startswith("w1_4_")])

feature_sets = {
    "membership_only_with_churn_prevented": MEMBERSHIP_FEATURES_WITH_CHURN,
    "membership_only_without_churn_prevented": MEMBERSHIP_FEATURES_WITHOUT_CHURN,
    "usage_w1_3_only": usage_w1_3_features,
    "usage_w1_4_only": usage_w1_4_features,
    "content_w1_3_only": content_w1_3_features,
    "content_w1_4_only": content_w1_4_features,
    "membership_plus_usage_w1_3_with_churn_prevented": MEMBERSHIP_FEATURES_WITH_CHURN + usage_w1_3_features,
    "membership_plus_usage_w1_3_without_churn_prevented": MEMBERSHIP_FEATURES_WITHOUT_CHURN + usage_w1_3_features,
    "membership_plus_usage_w1_4_with_churn_prevented": MEMBERSHIP_FEATURES_WITH_CHURN + usage_w1_4_features,
    "membership_plus_usage_w1_4_without_churn_prevented": MEMBERSHIP_FEATURES_WITHOUT_CHURN + usage_w1_4_features,
    "membership_plus_usage_content_w1_3_with_churn_prevented": MEMBERSHIP_FEATURES_WITH_CHURN + usage_w1_3_features + content_w1_3_features,
    "membership_plus_usage_content_w1_3_without_churn_prevented": MEMBERSHIP_FEATURES_WITHOUT_CHURN + usage_w1_3_features + content_w1_3_features,
    "membership_plus_usage_content_w1_4_with_churn_prevented": MEMBERSHIP_FEATURES_WITH_CHURN + usage_w1_4_features + content_w1_4_features,
    "membership_plus_usage_content_w1_4_without_churn_prevented": MEMBERSHIP_FEATURES_WITHOUT_CHURN + usage_w1_4_features + content_w1_4_features,
}

def build_dataset(window):
    if window == "w1_3":
        usage_by_mid, content_by_mid = usage13_by_mid, content13_by_mid
        usage_cols, content_cols = usage_w1_3_features, content_w1_3_features
    else:
        usage_by_mid, content_by_mid = usage14_by_mid, content14_by_mid
        usage_cols, content_cols = usage_w1_4_features, content_w1_4_features
    fields = METADATA_COLUMNS + [TARGET_COLUMN] + MEMBERSHIP_FEATURES_WITH_CHURN + usage_cols + content_cols
    rows = []
    missing_usage = 0
    missing_content = 0
    for mid in sorted(membership_by_mid, key=lambda x: int(x)):
        m = membership_by_mid[mid]
        u = usage_by_mid.get(mid)
        c = content_by_mid.get(mid)
        if u is None:
            missing_usage += 1
            u = {}
        if c is None:
            missing_content += 1
            c = {}
        out = {"membership_row_id": mid, "USER_KEY": m["USER_KEY"], TARGET_COLUMN: m[TARGET_COLUMN]}
        for col in MEMBERSHIP_FEATURES_WITH_CHURN:
            out[col] = m.get(col, "")
        for col in usage_cols:
            out[col] = u.get(col, "")
        for col in content_cols:
            out[col] = c.get(col, "")
        rows.append(out)
    return fields, rows, missing_usage, missing_content

w13_fields, w13_rows, w13_missing_usage, w13_missing_content = build_dataset("w1_3")
w14_fields, w14_rows, w14_missing_usage, w14_missing_content = build_dataset("w1_4")
write_csv(DATA_DIR / "modeling_dataset_v2_w1_3.csv", w13_rows, w13_fields)
write_csv(DATA_DIR / "modeling_dataset_v2_w1_4.csv", w14_rows, w14_fields)

feature_sets_payload = {
    "target_column": TARGET_COLUMN,
    "id_metadata_columns": ["membership_row_id"],
    "group_metadata_columns": ["USER_KEY"],
    "forbidden_features": sorted(FORBIDDEN_FEATURES),
    "categorical_features_to_encode_in_stage06": ["product_code", "is_promotion", "is_user_verified", "gender", "payment_device", "billing_method", "is_churn_prevented"] + [c for c in content_w1_3_features + content_w1_4_features if c.endswith("top_genre")],
    "feature_sets": feature_sets,
}
write_json(DATA_DIR / "feature_sets_v2.json", feature_sets_payload)

input_row_count_summary = [
    {"input_name": "membership_v2_preprocessed", "path": rel(STAGE02), "row_count": len(membership), "column_count": len(m_cols)},
    {"input_name": "usage_features_v2_w1_3", "path": rel(USAGE_W13), "row_count": len(usage13), "column_count": len(u13_cols)},
    {"input_name": "usage_features_v2_w1_4", "path": rel(USAGE_W14), "row_count": len(usage14), "column_count": len(u14_cols)},
    {"input_name": "content_features_v2_w1_3", "path": rel(CONTENT_W13), "row_count": len(content13), "column_count": len(c13_cols)},
    {"input_name": "content_features_v2_w1_4", "path": rel(CONTENT_W14), "row_count": len(content14), "column_count": len(c14_cols)},
]
write_csv(TABLE_DIR / "05_v2_input_row_count_summary.csv", input_row_count_summary, ["input_name", "path", "row_count", "column_count"])

merge_integrity = [
    {"window": "w1_3", "modeling_rows": len(w13_rows), "unique_membership_row_id": len({r["membership_row_id"] for r in w13_rows}), "missing_usage_rows": w13_missing_usage, "missing_content_rows": w13_missing_content, "status": "PASS" if len(w13_rows) == len(membership) else "FAIL"},
    {"window": "w1_4", "modeling_rows": len(w14_rows), "unique_membership_row_id": len({r["membership_row_id"] for r in w14_rows}), "missing_usage_rows": w14_missing_usage, "missing_content_rows": w14_missing_content, "status": "PASS" if len(w14_rows) == len(membership) else "FAIL"},
]
write_csv(TABLE_DIR / "05_v2_merge_integrity_summary.csv", merge_integrity, ["window", "modeling_rows", "unique_membership_row_id", "missing_usage_rows", "missing_content_rows", "status"])

def inventory_rows(window, fields):
    all_feature_cols = set()
    for features in feature_sets.values():
        all_feature_cols.update(features)
    out = []
    for col in fields:
        role = infer_role(col, all_feature_cols)
        if col in MEMBERSHIP_FEATURES_WITH_CHURN:
            source = "membership"
        elif col.startswith(window + "_") and "content" not in col and "genre" not in col and "release" not in col and "top_genre" not in col:
            source = "usage"
        elif col.startswith(window + "_"):
            source = "content"
        else:
            source = "metadata_or_target"
        out.append({"window": window, "column": col, "role": role, "source_family": source, "included_in_any_feature_set": "Y" if col in all_feature_cols else "N"})
    return out
column_inventory = inventory_rows("w1_3", w13_fields) + inventory_rows("w1_4", w14_fields)
write_csv(TABLE_DIR / "05_v2_modeling_column_inventory.csv", column_inventory, ["window", "column", "role", "source_family", "included_in_any_feature_set"])

forbidden_audit = []
for fs_name, cols in feature_sets.items():
    bad = [c for c in cols if has_forbidden_feature(c)]
    cross = []
    if "w1_3" in fs_name:
        cross = [c for c in cols if c.startswith("w1_4_")]
    if "w1_4" in fs_name:
        cross = [c for c in cols if c.startswith("w1_3_")]
    forbidden_audit.append({"feature_set": fs_name, "feature_count": len(cols), "forbidden_feature_count": len(bad), "forbidden_features": "|".join(bad), "cross_window_feature_count": len(cross), "cross_window_features": "|".join(cross), "status": "PASS" if not bad and not cross else "FAIL"})
write_csv(TABLE_DIR / "05_v2_forbidden_column_audit.csv", forbidden_audit, ["feature_set", "feature_count", "forbidden_feature_count", "forbidden_features", "cross_window_feature_count", "cross_window_features", "status"])

target_distribution = []
for window, rows in [("w1_3", w13_rows), ("w1_4", w14_rows)]:
    counts = Counter(r[TARGET_COLUMN] for r in rows)
    for target, count in sorted(counts.items()):
        target_distribution.append({"window": window, "target_value": target, "count": count, "rate": round(count/len(rows), 6)})
write_csv(TABLE_DIR / "05_v2_target_distribution_summary.csv", target_distribution, ["window", "target_value", "count", "rate"])

feature_set_summary = [{"feature_set": name, "feature_count": len(cols), "contains_is_churn_prevented": "Y" if "is_churn_prevented" in cols else "N", "window": "w1_3" if "w1_3" in name else "w1_4" if "w1_4" in name else "membership"} for name, cols in feature_sets.items()]
write_csv(TABLE_DIR / "05_v2_feature_set_summary.csv", feature_set_summary, ["feature_set", "feature_count", "contains_is_churn_prevented", "window"])

def missing_summary(window, rows, fields):
    out = []
    for col in fields:
        missing = sum(1 for r in rows if r.get(col, "") == "")
        out.append({"window": window, "column": col, "missing_count": missing, "missing_rate": round(missing/len(rows), 6)})
    return out
write_csv(TABLE_DIR / "05_v2_missing_value_summary.csv", missing_summary("w1_3", w13_rows, w13_fields) + missing_summary("w1_4", w14_rows, w14_fields), ["window", "column", "missing_count", "missing_rate"])

summary_payload = {
    "scope": "Stage 05 modeling dataset construction only. No model training.",
    "row_counts": {"w1_3": len(w13_rows), "w1_4": len(w14_rows)},
    "target_column": TARGET_COLUMN,
    "metadata_columns": METADATA_COLUMNS,
    "feature_set_count": len(feature_sets),
    "outputs": [rel(DATA_DIR / "modeling_dataset_v2_w1_3.csv"), rel(DATA_DIR / "modeling_dataset_v2_w1_4.csv"), rel(DATA_DIR / "feature_sets_v2.json"), rel(DATA_DIR / "modeling_dataset_summary.json")],
}
write_json(DATA_DIR / "modeling_dataset_summary.json", summary_payload)

report_path = DATA_DIR / "05_v2_modeling_dataset_report.md"
report_lines = [
    "# 05_v2 Modeling Dataset Report", "",
    "## Scope", "- Created final modeling tables only.", "- No model training, SHAP, segmentation, or business simulation was performed.", "",
    "## Outputs", f"- {rel(DATA_DIR / 'modeling_dataset_v2_w1_3.csv')}", f"- {rel(DATA_DIR / 'modeling_dataset_v2_w1_4.csv')}", f"- {rel(DATA_DIR / 'feature_sets_v2.json')}", f"- {rel(DATA_DIR / 'modeling_dataset_summary.json')}", "",
    "## Row Counts", f"- w1_3: {len(w13_rows):,} rows.", f"- w1_4: {len(w14_rows):,} rows.", "",
    "## Feature Policy", "- `is_repurchase` is target only.", "- `USER_KEY` is group metadata only.", "- `membership_row_id` is ID metadata only.", "- Categorical columns are not one-hot encoded here and are recorded for Stage 06 pipelines.",
]
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

raw_after = snapshot(RAW_FILES)
inputs_after = snapshot(INPUTS)
required_outputs = [DATA_DIR / "modeling_dataset_v2_w1_3.csv", DATA_DIR / "modeling_dataset_v2_w1_4.csv", DATA_DIR / "feature_sets_v2.json", DATA_DIR / "modeling_dataset_summary.json", report_path] + [TABLE_DIR / name for name in ["05_v2_input_row_count_summary.csv", "05_v2_merge_integrity_summary.csv", "05_v2_modeling_column_inventory.csv", "05_v2_forbidden_column_audit.csv", "05_v2_target_distribution_summary.csv", "05_v2_feature_set_summary.csv", "05_v2_missing_value_summary.csv"]]
all_forbidden_pass = all(row["status"] == "PASS" for row in forbidden_audit)
w13_cross_bad = any(c.startswith("w1_4_") for name, cols in feature_sets.items() if "w1_3" in name for c in cols)
w14_cross_bad = any(c.startswith("w1_3_") for name, cols in feature_sets.items() if "w1_4" in name for c in cols)
final_checks = [
    {"check": "raw_files_unchanged", "status": "PASS" if raw_before == raw_after else "FAIL", "detail": "raw file snapshots unchanged"},
    {"check": "no_project_root_data_output_created", "status": "PASS" if not (PROJECT_ROOT / "_data" / "02_interim" / "05_v2_modeling_dataset").exists() else "FAIL", "detail": "Stage 05 writes only under park.ingyeom/reports"},
    {"check": "stage02_stage03_stage04_outputs_not_overwritten", "status": "PASS" if inputs_before == inputs_after else "FAIL", "detail": "input snapshots unchanged"},
    {"check": "w1_3_row_count_23933", "status": "PASS" if len(w13_rows) == 23933 else "FAIL", "detail": f"rows={len(w13_rows)}"},
    {"check": "w1_4_row_count_23933", "status": "PASS" if len(w14_rows) == 23933 else "FAIL", "detail": f"rows={len(w14_rows)}"},
    {"check": "one_row_per_membership_row_id_w1_3", "status": "PASS" if len(w13_rows) == len({r["membership_row_id"] for r in w13_rows}) else "FAIL", "detail": "membership_row_id unique"},
    {"check": "one_row_per_membership_row_id_w1_4", "status": "PASS" if len(w14_rows) == len({r["membership_row_id"] for r in w14_rows}) else "FAIL", "detail": "membership_row_id unique"},
    {"check": "is_repurchase_target_not_feature", "status": "PASS" if all(TARGET_COLUMN not in cols for cols in feature_sets.values()) else "FAIL", "detail": "target excluded from feature_sets"},
    {"check": "USER_KEY_group_metadata_not_feature", "status": "PASS" if all("USER_KEY" not in cols for cols in feature_sets.values()) else "FAIL", "detail": "USER_KEY excluded from feature_sets"},
    {"check": "membership_row_id_id_metadata_not_feature", "status": "PASS" if all("membership_row_id" not in cols for cols in feature_sets.values()) else "FAIL", "detail": "membership_row_id excluded from feature_sets"},
    {"check": "no_forbidden_columns_in_feature_sets", "status": "PASS" if all_forbidden_pass else "FAIL", "detail": "see forbidden column audit"},
    {"check": "w1_3_feature_sets_contain_no_w1_4_features", "status": "PASS" if not w13_cross_bad else "FAIL", "detail": "cross-window prefix audit"},
    {"check": "w1_4_feature_sets_contain_no_w1_3_features", "status": "PASS" if not w14_cross_bad else "FAIL", "detail": "cross-window prefix audit"},
    {"check": "feature_sets_json_created", "status": "PASS" if (DATA_DIR / "feature_sets_v2.json").exists() else "FAIL", "detail": rel(DATA_DIR / "feature_sets_v2.json")},
    {"check": "no_model_trained", "status": "PASS", "detail": "No model training code or output"},
    {"check": "all_required_outputs_created", "status": "PASS" if all(p.exists() for p in required_outputs) else "FAIL", "detail": f"required_outputs={len(required_outputs)}"},
]
write_csv(TABLE_DIR / "05_v2_final_checks.csv", final_checks, ["check", "status", "detail"])

print("05_v2 modeling dataset construction completed.")
for row in final_checks:
    print(f"{row['check']}: {row['status']} - {row['detail']}")


05_v2 modeling dataset construction completed.
raw_files_unchanged: PASS - raw file snapshots unchanged
no_project_root_data_output_created: PASS - Stage 05 writes only under park.ingyeom/reports
stage02_stage03_stage04_outputs_not_overwritten: PASS - input snapshots unchanged
w1_3_row_count_23933: PASS - rows=23933
w1_4_row_count_23933: PASS - rows=23933
one_row_per_membership_row_id_w1_3: PASS - membership_row_id unique
one_row_per_membership_row_id_w1_4: PASS - membership_row_id unique
is_repurchase_target_not_feature: PASS - target excluded from feature_sets
USER_KEY_group_metadata_not_feature: PASS - USER_KEY excluded from feature_sets
membership_row_id_id_metadata_not_feature: PASS - membership_row_id excluded from feature_sets
no_forbidden_columns_in_feature_sets: PASS - see forbidden column audit
w1_3_feature_sets_contain_no_w1_4_features: PASS - cross-window prefix audit
w1_4_feature_sets_contain_no_w1_3_features: PASS - cross-window prefix audit
feature_sets_json_created: PAS